# Vision-Bot: Control gestual por Bluetooth

## Dr. Jesús Emmanuel Solís Pérez
### jsolisp@unam.mx

---

## DESCRIPCIÓN GENERAL: 
Visión-Bot es un taller intensivo y práctico diseñado para fusionar el poder de la Inteligencia Artificial con la robótica móvil. En este curso introductorio, los participantes construirán un sistema de control gestual desde cero. Aprenderán a utilizar la cámara de una computadora para capturar los movimientos de la mano en tiempo real, procesar los datos con Python y enviar comandos inalámbricos vía Bluetooth para controlar los motores de un carrito robótico.

# CONTENIDO DEL CURSO
## TEMARIO

1. Visión por computadora.
    * Uso de Python y MediaPipe para detectar los puntos de referencia de la mano.
2. Lógica de programación.
    * Clasificación de gestos simples.
3. Microcontroladores.
    * Configuración de un ESP32 para recibir comandos via Bluetooth.
4. Integración.
    * Envío de datos vía Bluetooth desde Python al carrito.
    * Pruebas de campo.

## HABILIDADES QUE APRENDERÁ

1. Uso y configuración de la librería MediaPipe para la detección de marcadores de la mano.
2. Normalización e interpretación de coordenadas espaciales en un flujo de video en vivo.
3. Estructuración de condicionales para la clasificación de gestos.
4. Transmisión de datos por Bluetooth.
5. Programación y configuración básica de microcontroladores.

## Requisitos de Software
1. **Python 3.8+**
2. **OpenCV** (para manipulación de video)
3. **Mediapipe**

---

In [1]:
# Importar los módulos necesarios de MediaPipe

import cv2 
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1787006301.975232  164494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787006301.975773  164494 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787006302.030174  164494 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787006303.947569  164494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

In [2]:
# Módulos de MediaPipe para la gestión de conexiones, dibujo y estilos predeterminados
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

# Constantes de configuración visual para el texto y las etiquetas
MARGIN = 10  
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # Color verde vibrante en formato RGB

# Variable global para almacenar el resultado más reciente del streaming
latest_detection_result = None

def print_result(result, output_image: mp.Image, timestamp_ms: int):
    """Callback asíncrono que recibe los resultados del livestream."""
    global latest_detection_result
    latest_detection_result = result

def draw_landmarks_on_image(rgb_image, detection_result):
    if detection_result is None or not detection_result.hand_landmarks:
        return rgb_image

    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    
    annotated_image = np.copy(rgb_image)

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        handedness = handedness_list[idx]

        # Dibuja los puntos clave y conexiones
        mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

        # Coordenadas para el texto de la lateralidad
        height, width, _ = annotated_image.shape
        x_coordinates = [landmark.x for landmark in hand_landmarks]
        y_coordinates = [landmark.y for landmark in hand_landmarks]
        text_x = int(min(x_coordinates) * width)
        text_y = int(min(y_coordinates) * height) - MARGIN

        cv2.putText(annotated_image, f"{handedness[0].category_name}",
                    (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                    FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

    return annotated_image

In [ ]:
# --- CONFIGURACIÓN DE LIVESTREAM ---
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    num_hands=4,
    result_callback=print_result)

# Inicializar la cámara web con OpenCV
cap = cv2.VideoCapture(0)

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("No se pudo acceder a la cámara.")
            break

        # OpenCV lee en BGR, MediaPipe requiere RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Obtener el timestamp actual en milisegundos (obligatorio para LIVE_STREAM)
        timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)

        # Envía la imagen de manera asíncrona al modelo
        landmarker.detect_async(mp_image, timestamp_ms)

        # Dibuja los resultados almacenados en la última respuesta del callback
        annotated_frame = draw_landmarks_on_image(rgb_frame, latest_detection_result)

        # Convierte de vuelta a BGR para mostrar con OpenCV
        bgr_annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)

        cv2.imshow('Adquisición Online', bgr_annotated_frame)

        # Presiona 'q' para salir del bucle
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787006305.946152  164600 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787006305.959380  164600 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/plugins"
W0000 00:00:1787006306.571495  164603 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontc